# Feature Selection

**Objetivo**: Validación y selección de *features*, evitando *data leakage* y preparando el conjunto **Top-60** para el entrenamiento del modelo de predicción del MERVAL.


## Tabla de contenidos

1. [Setup & Config](#setup-config)  
2. [Carga de dataset de ingeniería](#carga-dataset)  
3. [Creación del *target* `merval_apertura_+2`](#creacion-target)  
4. [Partición temporal y *helpers*](#tscv-helpers)  
5. [C-Serie: Filtros de validación](#c-serie)  
   - C1. *Missing values*  
   - C2. *Low variance*  
   - C3. *Clustering de correlación* + *Mutual Information*  
6. [Proceso en paralelo: Relevancias](#paralelo-relevancias)  
   - C4. XGBoost  
   - C5. Boruta  
   - C6. SHAP  
7. [Selecciones Top-K y Top-60 final](#selecciones)  
8. [Export de artefactos](#export)  
9. [Appendix: Utils](#appendix)


### Config  <a id='setup-config'></a>

- Configurar rutas de entrada/salida.
- Definir prefijos/columnas *mandatorias* (todas las `merval_*` salvo el *target*).
- Asegurar semillas reproducibles.
- Evitar *data leakage*: toda métrica basada en *target* se estima **sólo** en *folds de entrenamiento* con `TimeSeriesSplit`.


In [76]:
#!pip install boruta
#!pip install xgboost
#!pip install shap
#!pip install nbformat

In [77]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

# Modelos / métodos
try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except Exception as _e:
    _HAS_XGB = False
    print("[WARN] xgboost no disponible. Instalar con: pip install xgboost")

try:
    from boruta import BorutaPy
    from sklearn.ensemble import RandomForestRegressor
    _HAS_BORUTA = True
except Exception as _e:
    _HAS_BORUTA = False
    print("[WARN] boruta no disponible. Instalar con: pip install boruta")

try:
    import shap
    _HAS_SHAP = True
except Exception as _e:
    _HAS_SHAP = False
    print("[WARN] shap no disponible. Instalar con: pip install shap")

DATASET_NAME = "dataset"

# Ruta del dataset original :
# DATA_PATH = f"""../inputs/{DATASET_NAME}.csv"""

# Ruta del dataset con features generadas
INPUT_DATA_FILE = f"""{DATASET_NAME}_pre_modelo.csv"""

# Índice temporal
TIME_COL_CANDIDATES = ["date", "fecha"]

# Semillas
SEED = 32
rng = np.random.default_rng(SEED)

# Prefijo de *features* mandatorias (no se dropean en C1–C3)
MANDATORY = False
MANDATORY_PREFIX = "merval_"
MANDATORY_EXACTS = []

# Parámetros de selección
MAX_MISSING_FRAC     = 0.01   # C1
CONST_DOMINANCE_THR  = 0.99   # C2
CORR_CLUSTER_THR     = 0.95   # C3 (|corr| >= 0.95)

# CV temporal
N_SPLITS = 5
TSCV = TimeSeriesSplit(n_splits=N_SPLITS)

# Top-K para cada método
TOPK_PER_METHOD = 30

# Nombre de la columna base de apertura que usaremos para construir el target
TARGET_BASE = "merval_apertura"
TARGET = "merval_apertura_+2"

# Columnas a dropear
DROP_COL = []#["merval_cierre", "merval_maximo", "merval_minimo"]

# Utils de log
def _stamp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

### Input - Carga de datos  <a id='carga-dataset'></a>

- Buscar automáticamente el dataset exportado por `01_feature_engineering.ipynb`.
- Si no se encuentra, especificar `INPUT_DATA_FILE` en la celda de *Config*.


In [78]:
import nbformat as nbf
import pandas as pd

def _try_read_table(path: Path):
    if not path.exists():
        return None
    if path.suffix.lower() in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if path.suffix.lower() in (".feather", ".ft"):
        return pd.read_feather(path)
    if path.suffix.lower() in (".csv",):
        return pd.read_csv(path)
    return None

df = _try_read_table(Path(INPUT_DATA_FILE))

if df is None:
    raise FileNotFoundError("No se encontró el dataset. Edite INPUT_DATA_FILE en la celda de Config para apuntar al archivo correcto.")

# Normalizar índice temporal si existe
def _to_datetime_index(idx):
    if isinstance(idx, pd.DatetimeIndex):
        return idx
    try:
        return pd.to_datetime(idx)
    except Exception:
        return pd.Index(idx)

# Detectar columna temporal para setear como índice (si no lo está)
if not isinstance(df.index, pd.DatetimeIndex):
    for cand in TIME_COL_CANDIDATES:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            if df[cand].notna().sum() >= int(0.9 * len(df)):
                df = df.set_index(cand).sort_index()
                break

df.index = _to_datetime_index(df.index)
df = df.sort_index()

df = df.drop(columns=DROP_COL, errors="ignore")

_stamp(f"df shape: {df.shape}")
df.head(3)


[20:11:16] df shape: (77, 1449)


,merval_cierre,merval_maximo,emae_tendencia_ciclo_ma_5,badlar_ma_3,emae_desestacionalizado_lag1,inflacion_lag2,badlar_ma_5,tc_mayorista_std20,base_monetaria_std20,emae_tendencia_ciclo_lag2,...,diario_La Nación_prop_true_lag3,diario_La Nación_sum_true_lag3,diario_Pagina 12_prop_true_lag3,diario_Pagina 12_sum_true_lag3,categoria_fuente_periodistica_prop_true_lag3,categoria_fuente_periodistica_sum_true_lag3,categoria_fuente_analisis_prop_true_lag3,categoria_fuente_analisis_sum_true_lag3,categoria_fuente_oficial_prop_true_lag3,categoria_fuente_oficial_sum_true_lag3
fecha,,,,,,,,,,,,,,,,,,,,,
2025-01-06,2801220.75,2828290.0,150.376118,32.166667,152.097748,2.2,32.2125,7.104426,1.540313e+06,150.631781,...,0.419118,57.0,0.080882,11.0,1.0,136.0,0.0,0.0,0.0,0.0
2025-01-07,2822174.50,2867770.0,150.503949,32.375000,152.097748,2.2,32.1250,7.205263,1.601207e+06,150.631781,...,0.329412,28.0,0.070588,6.0,1.0,85.0,0.0,0.0,0.0,0.0
2025-01-08,2782477.00,2831310.0,150.631781,31.854167,152.097748,2.2,31.9125,7.203253,1.639175e+06,150.631781,...,0.439394,29.0,0.015152,1.0,1.0,66.0,0.0,0.0,0.0,0.0


In [79]:
df['merval_apertura_+2']

fecha
2025-01-06    198360.4844
2025-01-07    201379.7500
2025-01-08    213794.2500
2025-01-09    213794.2500
2025-01-10    215724.7031
                 ...     
2025-04-24    281827.4688
2025-04-25    290890.5313
2025-04-28    299421.7500
2025-04-29    303058.3750
2025-04-30    304796.9688
Name: merval_apertura_+2, Length: 77, dtype: float64

In [80]:
df[TARGET]

fecha
2025-01-06    198360.4844
2025-01-07    201379.7500
2025-01-08    213794.2500
2025-01-09    213794.2500
2025-01-10    215724.7031
                 ...     
2025-04-24    281827.4688
2025-04-25    290890.5313
2025-04-28    299421.7500
2025-04-29    303058.3750
2025-04-30    304796.9688
Name: merval_apertura_+2, Length: 77, dtype: float64

## 3) Creación del *target* `merval_apertura_+1`  <a id='creacion-target'></a>

- Detectar la columna de apertura del MERVAL (`merval_apertura` o similar).
- Construir el *target* como valor **+1 paso** (*shift* hacia el futuro): `target = apertura.shift(-1)`.
- Evitar *leakage*: el *shift* se aplica **después** de alinear el índice temporal.


In [81]:
TARGET_BASE

'merval_apertura'

In [82]:
TARGET

'merval_apertura_+2'

In [83]:
# Crear target +2
TARGET_COL = "merval_apertura_+2"
# df[TARGET_COL] = df[TARGET_BASE].shift(-2) # ya creado en el series temporales

# Quitar la última fila sin target (por el shift)
# df = df.iloc[:-2, :].copy()
# _stamp(f"Target +2 creado a partir de '{TARGET_BASE}'. df shape: {df.shape}")

# Separar X, y
y = df[TARGET_COL].copy()
X = df.drop(columns=[TARGET_COL]).copy()

Stamp = _stamp
Stamp(f"X shape: {X.shape} | y shape: {y.shape}")

[20:11:16] X shape: (77, 1448) | y shape: (77,)


In [84]:
X

,merval_cierre,merval_maximo,emae_tendencia_ciclo_ma_5,badlar_ma_3,emae_desestacionalizado_lag1,inflacion_lag2,badlar_ma_5,tc_mayorista_std20,base_monetaria_std20,emae_tendencia_ciclo_lag2,...,diario_La Nación_prop_true_lag3,diario_La Nación_sum_true_lag3,diario_Pagina 12_prop_true_lag3,diario_Pagina 12_sum_true_lag3,categoria_fuente_periodistica_prop_true_lag3,categoria_fuente_periodistica_sum_true_lag3,categoria_fuente_analisis_prop_true_lag3,categoria_fuente_analisis_sum_true_lag3,categoria_fuente_oficial_prop_true_lag3,categoria_fuente_oficial_sum_true_lag3
fecha,,,,,,,,,,,,,,,,,,,,,
2025-01-06,2801220.75,2828290.0,150.376118,32.166667,152.097748,2.2,32.2125,7.104426,1.540313e+06,150.631781,...,0.419118,57.0,0.080882,11.0,1.0,136.0,0.0,0.0,0.0,0.0
2025-01-07,2822174.50,2867770.0,150.503949,32.375000,152.097748,2.2,32.1250,7.205263,1.601207e+06,150.631781,...,0.329412,28.0,0.070588,6.0,1.0,85.0,0.0,0.0,0.0,0.0
2025-01-08,2782477.00,2831310.0,150.631781,31.854167,152.097748,2.2,31.9125,7.203253,1.639175e+06,150.631781,...,0.439394,29.0,0.015152,1.0,1.0,66.0,0.0,0.0,0.0,0.0
2025-01-09,2829730.50,2843740.0,150.631781,31.520833,152.097748,2.2,31.8750,7.255308,1.646088e+06,150.631781,...,0.415385,54.0,0.007692,1.0,1.0,130.0,0.0,0.0,0.0,0.0
2025-01-10,2805139.75,2839300.0,150.631781,31.375000,152.097748,2.2,31.7875,7.371175,1.634403e+06,150.631781,...,0.448529,61.0,0.058824,8.0,1.0,136.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-24,2232745.25,2331500.0,151.645380,34.000000,152.385915,2.8,34.9000,44.572587,2.090040e+06,151.645380,...,0.552846,68.0,0.024390,3.0,1.0,123.0,0.0,0.0,0.0,0.0
2025-04-25,2225242.75,2257680.0,151.645380,33.791667,152.385915,2.8,34.1000,47.008713,2.027912e+06,151.645380,...,0.511111,69.0,0.059259,8.0,1.0,135.0,0.0,0.0,0.0,0.0
2025-04-28,2179248.75,2242750.0,151.645380,33.500000,152.385915,2.8,33.8000,48.074391,1.953736e+06,151.645380,...,0.506849,74.0,0.075342,11.0,1.0,146.0,0.0,0.0,0.0,0.0


In [85]:
y

fecha
2025-01-06    198360.4844
2025-01-07    201379.7500
2025-01-08    213794.2500
2025-01-09    213794.2500
2025-01-10    215724.7031
                 ...     
2025-04-24    281827.4688
2025-04-25    290890.5313
2025-04-28    299421.7500
2025-04-29    303058.3750
2025-04-30    304796.9688
Name: merval_apertura_+2, Length: 77, dtype: float64

### Partición temporal y *helpers*  <a id='tscv-helpers'></a>

- `TimeSeriesSplit` con `n_splits=5` por defecto.
- Funciones auxiliares para:
  - Cálculo de *mutual information* promedio en *folds* (entrenamiento).
  - Importancias XGBoost promedio en *folds* (entrenamiento).
  - SHAP valores absolutos medios en *folds* (entrenamiento).  
  - Boruta (wrapper RF) sobre *fold* final (opción reproducible y eficiente).


In [86]:
def is_mandatory(col: str) -> bool:
    if col in MANDATORY_EXACTS:
        return True
    return col.startswith(MANDATORY_PREFIX)

def cv_mutual_info(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    mi_vals = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)
    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        # StandardScaler opcional (no requerido para MI)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mi = mutual_info_regression(Xtr.fillna(0), ytr.values, random_state=SEED)
        mi_vals += np.nan_to_num(mi)
        counts += 1
    mi_vals = np.divide(mi_vals, counts, out=np.zeros_like(mi_vals), where=counts>0)
    return pd.Series(mi_vals, index=X.columns).sort_values(ascending=False)

def cv_xgb_importance(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    if not _HAS_XGB:
        return pd.Series(0.0, index=X.columns)
    importances = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)
    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        model = XGBRegressor(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(Xtr.fillna(0), ytr.values)
        try:
            imp = model.feature_importances_
        except Exception:
            imp = np.zeros(X.shape[1], dtype=float)
        importances += np.nan_to_num(imp)
        counts += 1
    importances = np.divide(importances, counts, out=np.zeros_like(importances), where=counts>0)
    return pd.Series(importances, index=X.columns).sort_values(ascending=False)

def fold_shap_importance(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """SHAP mean(|value|) promedio en folds de entrenamiento con XGBRegressor.
    Si SHAP no está disponible, devuelve ceros.
    """
    if not (_HAS_XGB and _HAS_SHAP):
        return pd.Series(0.0, index=X.columns)

    shap_vals_accum = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)

    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        model = XGBRegressor(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED + fold,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(Xtr.fillna(0), ytr.values)

        # SHAP sobre una muestra del set de entrenamiento (para performance)
        sample_n = min(2000, len(Xtr))
        X_sample = Xtr.sample(n=sample_n, random_state=SEED).fillna(0)

        try:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(X_sample)
            shap_mean_abs = np.mean(np.abs(shap_vals), axis=0)
        except Exception:
            shap_mean_abs = np.zeros(X.shape[1], dtype=float)

        shap_vals_accum += shap_mean_abs
        counts += 1

    shap_vals_accum = np.divide(shap_vals_accum, counts, out=np.zeros_like(shap_vals_accum), where=counts>0)
    return pd.Series(shap_vals_accum, index=X.columns).sort_values(ascending=False)

def boruta_selection(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Corre Boruta (RF) en el último fold de entrenamiento y devuelve un puntaje normalizado por z-score
    donde *selected* recibe +1, *tentative* +0.5, *rejected* 0. Si Boruta no está disponible: ceros.
    """
    if not _HAS_BORUTA:
        return pd.Series(0.0, index=X.columns)

    # Tomar el último fold (más reciente) para un *snapshot* representativo
    folds = list(TSCV.split(X))
    tr_idx, _ = folds[-1]
    Xtr, ytr = X.iloc[tr_idx], y.iloc[tr_idx]

    rf = RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        n_jobs=-1,
        random_state=SEED,
        oob_score=False,
    )
    boruta = BorutaPy(
        estimator=rf,
        n_estimators="auto",
        alpha=0.05,
        two_step=True,
        max_iter=100,
        random_state=SEED
    )
    try:
        boruta.fit(Xtr.fillna(0).values, ytr.values)
        status = boruta.support_.astype(int) + 0.5 * boruta.support_weak_.astype(int)
        return pd.Series(status, index=X.columns).sort_values(ascending=False)
    except Exception as e:
        print(f"[WARN] Boruta falló: {e}")
        return pd.Series(0.0, index=X.columns)


### C-Serie: Filtros de validación  <a id='c-serie'></a>

**Alcance**: Se aplican a todas las *features* **no mandatorias**.

- **C1** (*missing values*): se dropean *features* con `frac_missing > 1%`.
- **C2** (*low variance / constante*): se dropean *features* donde un solo valor domina `≥ 99%` de las filas.
- **C3** (*clustering por correlación*): se forman *clusters* con `|corr| ≥ 0.95` y se retienen **Top-2** de cada cluster
  según *Mutual Information* promedio en CV (Cálculo de MI sólo con *folds de entrenamiento*).


In [87]:
# C1: Missing values 
if MANDATORY:
    feat_cols = [c for c in X.columns if not is_mandatory(c)]  # sólo procesables
else:
    feat_cols = X.columns.tolist()  # todos

miss_frac = X[feat_cols].isna().mean().sort_values(ascending=False)
drop_c1 = miss_frac[miss_frac > MAX_MISSING_FRAC].index.tolist()

# C2: Low variance (valor dominante)
def dominant_ratio(s: pd.Series) -> float:
    if s.empty:
        return 0.0
    vc = s.value_counts(dropna=False)
    if len(vc) == 0:
        return 0.0
    return float(vc.iloc[0]) / float(len(s))

dom_ratios = X[feat_cols].apply(dominant_ratio)
drop_c2 = dom_ratios[dom_ratios >= CONST_DOMINANCE_THR].index.tolist()

# Aplicar drops C1 y C2
drop_c12 = sorted(set(drop_c1 + drop_c2))
if MANDATORY:
    keep_after_c12 = [c for c in X.columns if c not in drop_c12 or is_mandatory(c)]
else:
    keep_after_c12 = [c for c in X.columns if c not in drop_c12]
X_c12 = X[keep_after_c12].copy()

_stamp(f"C1 drop: {len(drop_c1)} | C2 drop: {len(drop_c2)} | Total únicos C1–C2: {len(drop_c12)}")
_stamp(f"Shape tras C1–C2: {X_c12.shape}")

# C3: Correlation clustering + MI Top-2
# Matriz de correlación en todo el conjunto (no usa target => sin leakage)
corr = X_c12.select_dtypes(include=[np.number]).corr().abs()

# Construir clusters por umbral
visited = set()
clusters = []
cols_numeric = corr.columns.tolist()

for col in cols_numeric:
    if col in visited:
        continue
    # cluster = todos los nodos conectados con |corr| >= thr
    related = set([col])
    frontier = [col]
    while frontier:
        cur = frontier.pop()
        visited.add(cur)
        neighbors = corr.index[(corr[cur] >= CORR_CLUSTER_THR) & (corr.index != cur)].tolist()
        for nb in neighbors:
            if nb not in related:
                related.add(nb)
                if nb not in visited:
                    frontier.append(nb)
    clusters.append(sorted(list(related)))

# Calcular MI promedio por fold sólo sobre features candidatas (excluye mandatorias)
mi_scores = cv_mutual_info(X_c12[cols_numeric], y)

# En cada cluster, seleccionar Top-2 por MI (si hay menos, se retienen todas)
selected_c3 = set()
for cl in clusters:
    cl_sorted = sorted(cl, key=lambda c: mi_scores.get(c, 0.0), reverse=True)
    top2 = cl_sorted[:3]
    for c in top2:
        selected_c3.add(c)

# Componer columnas finales tras C3
if MANDATORY:
    keep_c3 = sorted(set([c for c in X_c12.columns if is_mandatory(c)]) | selected_c3 | set([c for c in X_c12.columns if c not in cols_numeric]))
else:
    keep_c3 = sorted(set(selected_c3) | set([c for c in X_c12.columns if c not in cols_numeric]))
X_c3 = X_c12[keep_c3].copy()
_stamp(f"Clusters C3: {len(clusters)} | Seleccionadas por C3: {len(selected_c3)} | Shape X_c3: {X_c3.shape}")


[20:11:16] C1 drop: 8 | C2 drop: 22 | Total únicos C1–C2: 26
[20:11:16] Shape tras C1–C2: (77, 1422)
[20:11:20] Clusters C3: 1153 | Seleccionadas por C3: 1411 | Shape X_c3: (77, 1411)


### Proceso en paralelo: Relevancias  <a id='paralelo-relevancias'></a>

Se calculan **tres** rankings independientes sobre `X_c3`:

- **C4 – XGBoost**: promedio de importancias en *folds*.
- **C5 – Boruta (RF)**: *selected=1*, *weak=0.5*, *rejected=0* sobre el último *fold* de entrenamiento.
- **C6 – SHAP (XGB)**: media de `|SHAP|` en muestra del set de entrenamiento por *fold*.

> Nota: si alguna librería no está instalada, el ranking correspondiente será cero para todas las variables.


In [92]:
# Columnas numéricas (XGBoost/SHAP requieren numérico)
numeric_cols = X_c3.select_dtypes(include=[np.number]).columns.tolist()
Xc3_num = X_c3[numeric_cols].copy()

# C4: XGBoost
xgb_rank = cv_xgb_importance(Xc3_num, y).rename("xgb_importance")

# C5: Boruta
boruta_rank = boruta_selection(Xc3_num, y).rename("boruta_score")

# C6: SHAP
shap_rank = fold_shap_importance(Xc3_num, y).rename("shap_mean_abs")

# Ensamble de rankings en un único DataFrame
rank_df = pd.concat([xgb_rank, boruta_rank, shap_rank], axis=1).fillna(0.0)
rank_df["avg_norm_rank"] = 0.0

# Normalización por columna (min-max) antes de promediar
for col in ["xgb_importance", "boruta_score", "shap_mean_abs"]:
    vals = rank_df[col].values
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if vmax > vmin:
        rank_df[col + "_norm"] = (vals - vmin) / (vmax - vmin)
    else:
        rank_df[col + "_norm"] = 0.0
rank_df["avg_norm_rank"] = rank_df[[c for c in rank_df.columns if c.endswith("_norm")]].mean(axis=1)

rank_df = rank_df.sort_values("avg_norm_rank", ascending=False)
_stamp(f"Ranking combinado (shape={rank_df.shape})")
rank_df.head(10)

[20:12:06] Ranking combinado (shape=(1411, 7))


,xgb_importance,boruta_score,shap_mean_abs,avg_norm_rank,xgb_importance_norm,boruta_score_norm,shap_mean_abs_norm
merval_maximo_diff10,0.038860,1.0,1399.828290,0.697606,0.504554,1.0,0.588262
merval_maximo_hp_ema20,0.031699,1.0,1593.279346,0.693712,0.411577,1.0,0.669558
base_monetaria_std20,0.077019,0.0,2379.599317,0.666667,1.000000,0.0,1.000000
merval_maximo_pct10,0.052635,1.0,419.944885,0.619957,0.683395,1.0,0.176477
merval_apertura_pct10,0.037366,1.0,570.237964,0.574927,0.485146,1.0,0.239636
merval_apertura_std20,0.034744,1.0,166.206850,0.506984,0.451107,1.0,0.069847
emae_desestacionalizado_hp_rm20,0.010633,1.0,375.413813,0.431940,0.138056,1.0,0.157763
merval_cierre_std20,0.005304,1.0,361.045367,0.406863,0.068863,1.0,0.151725
base_monetaria_diff10,0.046777,0.0,1407.320178,0.399583,0.607339,0.0,0.591411
tasa_prestamos_personales_ma_3,0.003946,1.0,255.013708,0.386132,0.051230,1.0,0.107167


### Selecciones Top-K y Top-60 final  <a id='selecciones'></a>

- **D**: Selección **Top-20** de cada método (XGB, Boruta, SHAP).  
- **E**: Completar **Top-60** tomando el *promedio normalizado* (`avg_norm_rank`).  
- Si MANDATORY, se preservan todas las *features mandatorias* (`merval_*`).  

In [93]:
# D) Top-20 por método
top20_xgb    = rank_df.sort_values("xgb_importance", ascending=False).head(TOPK_PER_METHOD).index.tolist()
top20_boruta = rank_df.sort_values("boruta_score",   ascending=False).head(TOPK_PER_METHOD).index.tolist()
top20_shap   = rank_df.sort_values("shap_mean_abs",  ascending=False).head(TOPK_PER_METHOD).index.tolist()

selected_d = list(dict.fromkeys(top20_xgb + top20_boruta + top20_shap))  # únicos manteniendo orden

# E) Completar Top-60 con promedio normalizado
if MANDATORY:
    mandatory_cols = [c for c in X_c3.columns if is_mandatory(c)]
else:
    mandatory_cols = []  # IGNORE
pool = [c for c in rank_df.index if c not in selected_d]
need_more = max(0, 60 - (len(selected_d) + len(mandatory_cols)))

extra = rank_df.loc[pool].sort_values("avg_norm_rank", ascending=False).head(need_more).index.tolist()

final_top60 = list(dict.fromkeys(mandatory_cols + selected_d + extra))[:60]

_stamp(f"Mandatorias: {len(mandatory_cols)} | D únicos: {len(selected_d)} | Extra: {len(extra)} | Top-60 final: {len(final_top60)}")

# Conjunto final de entrenamiento
cols_final = final_top60 + [c for c in X_c3.columns if is_mandatory(c) and c not in final_top60]
cols_final = list(dict.fromkeys(cols_final))  # por si acaso
X_final = X_c3[final_top60].copy()

_stamp(f"X_final shape: {X_final.shape}")
pd.DataFrame({"feature": final_top60}).head(10)


[20:12:16] Mandatorias: 0 | D únicos: 52 | Extra: 8 | Top-60 final: 60
[20:12:16] X_final shape: (77, 60)


,feature
0,base_monetaria_std20
1,act_luis_caputo_prop_no_cero_lag1
2,merval_maximo_pct10
3,base_monetaria_diff10
4,merval_maximo_diff10
5,merval_apertura_pct10
6,merval_apertura_std20
7,badlar_ma_3
8,merval_maximo_hp_ema20
9,inflacion_hp_rm20


In [94]:
rank_df.head(30)

,xgb_importance,boruta_score,shap_mean_abs,avg_norm_rank,xgb_importance_norm,boruta_score_norm,shap_mean_abs_norm
merval_maximo_diff10,0.038860,1.0,1399.828290,0.697606,0.504554,1.0,0.588262
merval_maximo_hp_ema20,0.031699,1.0,1593.279346,0.693712,0.411577,1.0,0.669558
base_monetaria_std20,0.077019,0.0,2379.599317,0.666667,1.000000,0.0,1.000000
merval_maximo_pct10,0.052635,1.0,419.944885,0.619957,0.683395,1.0,0.176477
merval_apertura_pct10,0.037366,1.0,570.237964,0.574927,0.485146,1.0,0.239636
merval_apertura_std20,0.034744,1.0,166.206850,0.506984,0.451107,1.0,0.069847
emae_desestacionalizado_hp_rm20,0.010633,1.0,375.413813,0.431940,0.138056,1.0,0.157763
merval_cierre_std20,0.005304,1.0,361.045367,0.406863,0.068863,1.0,0.151725
base_monetaria_diff10,0.046777,0.0,1407.320178,0.399583,0.607339,0.0,0.591411
tasa_prestamos_personales_ma_3,0.003946,1.0,255.013708,0.386132,0.051230,1.0,0.107167


In [95]:
final_top60

['base_monetaria_std20',
 'act_luis_caputo_prop_no_cero_lag1',
 'merval_maximo_pct10',
 'base_monetaria_diff10',
 'merval_maximo_diff10',
 'merval_apertura_pct10',
 'merval_apertura_std20',
 'badlar_ma_3',
 'merval_maximo_hp_ema20',
 'inflacion_hp_rm20',
 'impacto_mercosur_std',
 'badlar_ma_5',
 'act_luis_caputo_mean_lag1',
 'act_luis_caputo_std_lag1',
 'act_luis_caputo_prop_no_cero',
 'act_luis_caputo_std',
 'act_fondo_monetario_internacional_directorio_prop_no_cero_lag3',
 'menciona_sector_agroexportador_prop_true',
 'm1_std20',
 'emae_desestacionalizado_hp_rm20',
 'act_luis_caputo_mean',
 'impacto_sector_energia_std_lag1',
 'bonos_soberanos_mean_lag3',
 'volumen_mercado_mean_lag3',
 'act_fondo_monetario_internacional_directorio_mean_lag3',
 'caracter_prospectivo_sum_true_lag2',
 'tipo_evento_electoral_prop_true',
 'cant_tickers_std',
 'sector_Construcción / Inmobiliario_prop_true_lag2',
 'tipo_actor_principal_municipio_prop_true',
 'merval_cierre_std20',
 'tasa_prestamos_personales_

### Export de artefactos  <a id='export'></a>

Se exportan:

- `selected_features_top60.csv`: lista ordenada de *features*.
- `feature_ranking.csv`: tabla de *rankings* con normalizados y promedio.
- `dataset_top60.parquet`: dataset reducido a Top-60 + *target*.
- `feature_selection_report.md`: breve reporte con conteos y umbrales utilizados.


In [96]:
pd.concat([X_final, y.rename(TARGET)], axis=1)

,base_monetaria_std20,act_luis_caputo_prop_no_cero_lag1,merval_maximo_pct10,base_monetaria_diff10,merval_maximo_diff10,merval_apertura_pct10,merval_apertura_std20,badlar_ma_3,merval_maximo_hp_ema20,inflacion_hp_rm20,...,act_desconocido_mean,act_luis_caputo_mean_lag2,act_luis_caputo_mean_lag3,actividad_economica_mean_lag2,bonos_soberanos_mean_lag2,bonos_soberanos_mean,act_desconocido_std_lag1,tc_mayorista_pct1,menciona_sector_industrial_prop_true_lag1,merval_apertura_+2
fecha,,,,,,,,,,,,,,,,,,,,,
2025-01-06,1.540313e+06,0.000000,0.085341,3550135.0,222390.0,0.073022,174175.966568,32.166667,291013.576095,-0.425,...,0.076923,0.000000,0.000000,-0.033529,0.032941,0.043462,0.361298,0.000966,0.212121,198360.4844
2025-01-07,1.601207e+06,0.000000,0.069872,3313629.0,187290.0,0.080634,190296.837624,32.375000,299017.997420,-0.400,...,0.110294,0.000000,0.000000,0.013636,0.073485,0.020221,0.267500,0.000483,0.169231,201379.7500
2025-01-08,1.639175e+06,0.007353,0.111146,3395050.0,283210.0,0.119459,198707.818223,31.854167,237552.473856,-0.375,...,0.076023,0.000000,0.000000,-0.017692,0.043462,0.012281,0.314414,0.000483,0.139706,213794.2500
2025-01-09,1.646088e+06,0.017544,0.150017,2291561.0,370960.0,0.149834,198660.207128,31.520833,226174.619203,-0.350,...,0.111111,0.007353,0.000000,-0.002574,0.020221,0.014815,0.265814,0.000965,0.181287,213794.2500
2025-01-10,1.634403e+06,0.000000,0.125907,2042786.0,317510.0,0.146772,196788.086402,31.375000,200617.036422,-0.325,...,0.096154,0.017544,0.007353,-0.003216,0.012281,0.038942,0.315735,0.001927,0.148148,215724.7031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-24,2.090040e+06,0.093525,0.094889,-3578926.0,202060.0,0.105350,141445.788409,34.000000,33843.127925,-0.225,...,0.064000,0.007407,0.040650,0.009259,0.034074,0.029600,0.292220,0.048036,0.179856,281827.4688
2025-04-25,2.027912e+06,0.032000,0.028256,-2630057.0,62040.0,0.121632,135398.668462,33.791667,-36169.550925,-0.180,...,0.109589,0.093525,0.007407,0.006475,0.026259,0.022945,0.245738,-0.001913,0.160000,290890.5313
2025-04-28,1.953736e+06,0.025641,0.032859,-3289984.0,71350.0,0.019081,123209.631784,33.500000,-46232.927027,-0.135,...,0.073394,0.037500,0.020548,-0.015000,0.020000,-0.025688,0.321553,-0.009368,0.192308,299421.7500


In [97]:
# Archivos de salida
features_csv = f"{DATASET_NAME}_selected_features_llm_top60.csv"
ranking_csv  = f"{DATASET_NAME}_llm_feature_ranking.csv"
dataset_csv   = f"../{DATASET_NAME}_llm_fs_top60_+2.csv"
report_md    = f"{DATASET_NAME}_llm_feature_selection_report.md"

# Guardar
pd.DataFrame({"feature": final_top60}).to_csv(features_csv, index=False)
rank_df.to_csv(ranking_csv, index=True)
pd.concat([X_final, y.rename(TARGET_COL)], axis=1).to_csv(dataset_csv)

# Reporte rápido
report = f"""
# Feature Selection Report
Fecha: {datetime.now():%Y-%m-%d %H:%M:%S}

- Input: `{INPUT_DATA_FILE}`
- Filtrado C1 (missing > {MAX_MISSING_FRAC:.2%}): {len(drop_c1)}
- Filtrado C2 (dominancia >= {CONST_DOMINANCE_THR:.2%}): {len(drop_c2)}
- Clusters C3 (|corr| >= {CORR_CLUSTER_THR:.0%}): {len(clusters)}
- Mandatorias retenidas: {len(mandatory_cols)}
- Top-20 por método: XGB={len(top20_xgb)}, Boruta={len(top20_boruta)}, SHAP={len(top20_shap)}
- Top-60 final: {len(final_top60)}

Parámetros:
- N_SPLITS={N_SPLITS}
- TARGET={TARGET_COL}
"""

with open(report_md, "w", encoding="utf-8") as f:
    f.write(report)

_stamp(f"Exportado:\n - {features_csv}\n - {ranking_csv}\n - {dataset_csv}\n - {report_md}")

[20:14:35] Exportado:
 - dataset_selected_features_llm_top60.csv
 - dataset_llm_feature_ranking.csv
 - ../dataset_llm_fs_top60_+2.csv
 - dataset_llm_feature_selection_report.md
